# 12 — 2026-10-26: coding Qwen2.5-VL's explanations with the same two LLM judges

As notebook 07, for the explanations of experiment 11, with the same codes and prompt (`configs/2026-10-26_code_explanations_qwen25vl.toml`). Results: `outputs/2026-10-26_code_explanations_qwen25vl/judge_labels.jsonl`; finished rows are skipped.

In [9]:
N_QUESTIONS = None  # None = all 598

In [10]:
import hashlib, json, os, threading, tomllib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().parent
CFG = tomllib.loads((ROOT / "configs/2026-10-26_code_explanations_qwen25vl.toml").read_text())
SOURCE = tomllib.loads((ROOT / f"configs/{CFG['source_experiment']}.toml").read_text())
CODES = list(CFG["codes"])
JUDGES = CFG["judge_models"]
EXP_DIR = ROOT / "outputs" / CFG["experiment"]
EXP_DIR.mkdir(parents=True, exist_ok=True)
OUT = EXP_DIR / "judge_labels.jsonl"

expl = pd.read_csv(ROOT / "outputs" / CFG["source_experiment"] / "explanations.csv")
options = pd.read_parquet(ROOT / "data-private/processed/emotion_questions.parquet").set_index("question_id")
expl["key"] = expl.question_id + "|" + expl.condition
expl = expl[expl.order.isin(sorted(expl.order.unique())[:N_QUESTIONS])]


def options_text(row):
    texts = {L: options.loc[row.question_id, f"option_{L}"].strip() for L in "ABCD"} | {"E": SOURCE["option_e"]}
    return "\n".join(f"{shown}. {texts[original]}" for shown, original in zip("ABCDE", row.option_order))


expl["options_shown"] = [options_text(r) for r in expl.itertuples()]
expl["answer_text"] = [r.options_shown.split("\n")["ABCDE".index(r.answer_shown)] for r in expl.itertuples()]
print(f"{expl.question_id.nunique()} questions, {len(expl)} explanations; judges: {JUDGES} -> {OUT.relative_to(ROOT)}")

598 questions, 3588 explanations; judges: ['anthropic/claude-haiku-5.5', 'openai/gpt-6-luna'] -> outputs/2026-10-26_code_explanations_qwen25vl/judge_labels.jsonl


## Judges

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(ROOT / ".env")
client = OpenAI(base_url=CFG["api_base_url"], api_key=os.environ["OPENROUTER_API_KEY"])
SYSTEM = CFG["prompts"]["system"].format(codes="\n".join(f"- {c}: {d}" for c, d in CFG["codes"].items()))
PROMPT_SHA = hashlib.sha256(SYSTEM.encode()).hexdigest()[:12]
SCHEMA = {"type": "object",
          "properties": {"rationale": {"type": "string"}, **{c: {"type": "boolean"} for c in CODES}},
          "required": ["rationale"] + CODES, "additionalProperties": False}


def item_text(row):
    return CFG["prompts"]["item"].format(question=row.question.strip(), options=row.options_shown,
                                         answer=row.answer_text, explanation=row.explanation)


def judge(row, model):
    extra = {"seed": CFG["seed"]} if model in CFG["seed_models"] else {}
    response = client.chat.completions.create(
        model=model, max_tokens=CFG["max_tokens"],
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": item_text(row)}],
        response_format={"type": "json_schema", "json_schema": {"name": "codes", "strict": True, "schema": SCHEMA}},
        extra_body={"reasoning": {"effort": CFG["reasoning_effort"]}}, **extra,
    )
    choice = response.choices[0]
    labels = json.loads(choice.message.content)
    assert set(labels) == {"rationale"} | set(CODES), labels
    details = getattr(response.usage, "completion_tokens_details", None)
    return {"key": row.key, "model": model, **labels, "finish_reason": choice.finish_reason,
            "response_model": response.model, "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "reasoning_tokens": getattr(details, "reasoning_tokens", None), "prompt_sha": PROMPT_SHA}


def done():
    return {(r["key"], r["model"]) for r in map(json.loads, OUT.open())} if OUT.exists() else set()


print(SYSTEM)
print("\n===== example item")
print(item_text(expl.iloc[0]))

You code short explanations written by an AI model. The model answered a multiple-choice question about a person's emotions in a scene from a short film, and was then asked: "Explain in one sentence which information you based your answer on." You are not told what input the model had (it may or may not have had a transcript or a video).

Code only what the explanation itself claims, not whether the claim is true or the answer is right. Apply each code independently; several or none can apply.

Codes:
- cites_dialogue: The explanation quotes or paraphrases what a character says, or refers to a transcript as something the model had (for example "the transcript says ..." or "the transcript does not mention ..."). Subtitles and captions count as a transcript. Saying only that there is no dialogue, or that no transcript was provided, without referring to a transcript the model had, is NOT this code. Do not count speech that only repeats the wording of the question or of any answer option.


## Run

In [12]:
lock, errors = threading.Lock(), []

def work(task):
    row, model = task
    try:
        result = judge(row, model)
    except Exception as e:
        errors.append((row.key, model, repr(e)[:200]))
        return
    with lock, OUT.open("a") as f:
        f.write(json.dumps(result) + "\n")

finished = done()
tasks = [(row, model) for model in JUDGES for row in expl.itertuples() if (row.key, model) not in finished]
with ThreadPoolExecutor(CFG["n_workers"]) as pool:
    list(tqdm(pool.map(work, tasks), total=len(tasks), unit="label"))
print(f"done {len(tasks) - len(errors)}, failed {len(errors)}")
for e in errors[:10]:
    print(" ", *e)

  0%|          | 0/1 [00:00<?, ?label/s]

done 1, failed 0


## Check

In [13]:
labels = pd.read_json(OUT, lines=True)
labels = labels[labels.key.isin(expl.key)]
display(labels.groupby("model")[["finish_reason", "reasoning_tokens"]].agg(lambda s: s.value_counts(dropna=False).to_dict()))
labels["codes"] = labels[CODES].apply(lambda r: ", ".join(c for c in CODES if r[c]) or "-", axis=1)
side = labels.pivot(index="key", columns="model", values="codes")
side.columns = [m.split("/")[1] for m in side.columns]
pd.set_option("display.max_colwidth", None)
display(expl.head(30).set_index("key")[["explanation"]].join(side))

,finish_reason,reasoning_tokens
model,,
anthropic/claude-haiku-5.5,{'stop': 3588},"{0: 2177, 181: 15, 163: 14, 226: 13, 193: 13, 192: 12, 215: 12, 200: 12, 206: 11, 188: 10, 243: 10, 190: 10, 165: 10, 214: 10, 228: 10, 179: 10, 168: 9, 189: 9, 203: 9, 241: 9, 248: 9, 178: 9, 173: 9, 235: 9, 151: 9, 247: 9, 246: 9, 222: 9, 195: 8, 202: 8, 199: 8, 272: 8, 153: 8, 273: 8, 152: 8, 174: 8, 253: 8, 274: 8, 210: 8, 156: 8, 154: 8, 234: 8, 183: 7, 219: 7, 205: 7, 198: 7, 170: 7, 224: 7, 208: 7, 216: 7, 223: 7, 217: 7, 304: 7, 136: 7, 171: 7, 221: 7, 197: 7, 255: 7, 186: 7, 218: 7, 148: 7, 144: 7, 194: 7, 187: 7, 160: 7, 252: 7, 230: 7, 184: 7, 239: 7, 162: 6, 138: 6, 257: 6, 318: 6, 302: 6, 249: 6, 261: 6, 220: 6, 211: 6, 262: 6, 167: 6, 149: 6, 231: 6, 175: 6, 182: 6, 286: 6, 196: 5, 279: 5, 225: 5, 169: 5, 157: 5, 161: 5, 158: 5, 176: 5, 250: 5, 380: 5, 240: 5, 145: 5, 212: 5, 436: 5, 133: 5, ...}"
openai/gpt-6-luna,{'stop': 3588},"{109: 34, 140: 32, 113: 30, 117: 30, 133: 29, 122: 29, 127: 29, 111: 29, 103: 29, 128: 28, 108: 28, 146: 28, 116: 28, 119: 27, 98: 27, 81: 27, 107: 27, 132: 27, 123: 27, 115: 26, 78: 26, 142: 26, 121: 26, 90: 26, 85: 25, 95: 25, 138: 25, 118: 25, 141: 25, 143: 24, 101: 24, 83: 24, 110: 24, 144: 24, 131: 24, 86: 24, 126: 23, 135: 23, 97: 23, 100: 23, 80: 22, 104: 22, 136: 22, 92: 22, 125: 22, 120: 22, 149: 21, 76: 21, 94: 21, 87: 21, 159: 21, 96: 21, 82: 20, 112: 20, 93: 20, 72: 20, 105: 20, 158: 20, 161: 20, 106: 20, 114: 20, 89: 19, 134: 19, 88: 19, 157: 19, 91: 19, 71: 19, 152: 19, 84: 19, 147: 19, 129: 18, 151: 18, 77: 18, 124: 18, 148: 18, 130: 18, 166: 17, 150: 17, 137: 17, 162: 16, 99: 16, 102: 16, 179: 15, 75: 15, 185: 15, 183: 15, 167: 15, 74: 15, 172: 15, 178: 15, 154: 15, 165: 14, 189: 14, 168: 14, 176: 14, 70: 14, 190: 14, 182: 13, 177: 13, 188: 13, ...}"


,explanation,claude-haiku-5.5,gpt-6-luna
key,,,
FH-76|Q,"The answer is C because the question does not provide any specific details or expressions of emotion from the man regarding knocking on the door, making it impossible to determine his feelings from the given information.",says_insufficient,says_insufficient
FH-76|Q_noE,"The answer is C because the context implies that the man feels nervous about knocking on the door, suggesting a sense of apprehension or uncertainty.",-,-
FH-76|T,"The answer is C because the transcript provided no dialogue, so there is no information to determine how the man feels about knocking on the door.","cites_dialogue, says_insufficient","cites_dialogue, says_insufficient"
FH-76|T_told,The answer cannot be determined from the given transcript as it contains no dialogue or context about the man's feelings regarding knocking on the door.,"cites_dialogue, says_insufficient","cites_dialogue, says_insufficient"
FH-76|VT,"The video does not provide any explicit dialogue or facial expressions that indicate the man's feelings about knocking on the door, so it is not possible to determine his emotions from the given information.","says_insufficient, names_missing_visual","says_insufficient, names_missing_visual"
FH-76|VTmis,"The video does not provide any dialogue or visual cues indicating the man's feelings about knocking on the door, so it is impossible to determine his emotions from the given information.","says_insufficient, names_missing_visual","says_insufficient, names_missing_visual"
97rJU|Q,The answer cannot be determined from the given information as there is no context provided about the young woman's feelings or reactions towards the older woman's violent reaction.,says_insufficient,says_insufficient
97rJU|Q_noE,"The answer is C because the context implies that the younger woman is likely to step back as a defensive response to the older woman's violent reaction, indicating fear or an attempt to avoid harm.",-,-
97rJU|T,"The answer is C because the dialogue shows the younger woman repeatedly saying ""Get off"" and ""Get your black hands off,"" indicating fear and a desire to avoid further harm, which aligns with feeling scared and stepping back.",cites_dialogue,cites_dialogue


## Status

In [14]:
labels = pd.read_json(OUT, lines=True)
status = labels.groupby("model").agg(rows=("key", "nunique"), prompt_tokens=("prompt_tokens", "sum"),
                                      completion_tokens=("completion_tokens", "sum"))
status["cost_usd"] = (status.prompt_tokens * 0.10 + status.completion_tokens * 0.50) / 1e6
print(f"{len(expl)} explanations; prompt versions: {labels.prompt_sha.value_counts().to_dict()}")
display(status)

3588 explanations; prompt versions: {77044885371: 7176}


,rows,prompt_tokens,completion_tokens,cost_usd
model,,,,
anthropic/claude-haiku-5.5,3588,6256883,943744,1.09756
openai/gpt-6-luna,3588,3623633,1000794,0.86276


## Agreement between the judges

In [15]:
from sklearn.metrics import cohen_kappa_score

labels = pd.read_json(OUT, lines=True, dtype={"prompt_sha": str})
labels = labels[labels.key.isin(expl.key)]
wide = {m: labels[labels.model == m].set_index("key")[CODES].astype(bool).reindex(expl.key) for m in JUDGES}
a, b = wide[JUDGES[0]], wide[JUDGES[1]]
agreement = pd.DataFrame({"agree": (a == b).mean(), "kappa": [cohen_kappa_score(a[c], b[c]) for c in CODES],
                          **{f"yes_{m.split('/')[1]}": wide[m].sum() for m in JUDGES}})
agreement.to_csv(EXP_DIR / "agreement.csv")
print("explanations with any disagreement:", (a != b).any(axis=1).sum(), "of", len(expl))
agreement.round(3)

explanations with any disagreement: 438 of 3588


,agree,kappa,yes_claude-haiku-5.5,yes_gpt-6-luna
cites_dialogue,0.968,0.925,1156,1081
cites_visual,0.951,0.737,345,401
says_insufficient,0.996,0.991,2203,2201
names_missing_visual,0.946,0.784,494,558
notices_mismatch,0.997,0.469,11,6


## Codes by condition

% of explanations where both judges give the code.

In [16]:
conditions = [c for c in SOURCE["conditions"] if c in set(expl.condition)]
condition = expl.condition.values
by_condition = (a & b).groupby(condition).mean().reindex(conditions) * 100
by_condition["judges_differ"] = (a != b).any(axis=1).groupby(condition).mean().reindex(conditions) * 100
by_condition.to_csv(EXP_DIR / "codes_by_condition.csv")
audio = expl.explanation.str.contains(r"\baudio", case=False)
print('explanations mentioning "audio":', audio.groupby(condition).sum().reindex(conditions).to_dict())
by_condition.round(1)

explanations mentioning "audio": {'Q_noE': 0, 'Q': 1, 'T': 1, 'T_told': 1, 'VT': 9, 'VTmis': 34}


,cites_dialogue,cites_visual,says_insufficient,names_missing_visual,notices_mismatch,judges_differ
Q_noE,0.3,4.8,2.5,1.0,0.0,4.0
Q,0.0,0.0,97.3,7.7,0.0,2.2
T,43.3,0.3,81.4,7.4,0.0,4.0
T_told,81.3,0.0,86.1,11.5,0.0,6.5
VT,28.3,30.1,33.3,15.1,0.0,23.2
VTmis,24.2,12.4,66.2,29.1,0.7,33.3
